# Q4: Feature Engineering

**Phase 5:** Feature Engineering & Aggregation  
**Points: 9 points**

**Focus:** Create derived features, perform time-based aggregations, calculate rolling windows.

**Lecture Reference:** Lecture 11, Notebook 2 ([`11/demo/02_wrangling_feature_engineering.ipynb`](https://github.com/christopherseaman/datasci_217/blob/main/11/demo/02_wrangling_feature_engineering.ipynb)), Phase 5. Also see Lecture 09 (rolling windows).

---

## Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load wrangled data from Q3
# df = pd.read_csv('output/q3_wrangled_data.csv', parse_dates=['Measurement Timestamp'], index_col='Measurement Timestamp')
# Or if you saved without index:
df = pd.read_csv('output/q3_wrangled_data.csv')
df['Measurement Timestamp'] = pd.to_datetime(df['Measurement Timestamp'])
df = df.set_index('Measurement Timestamp')
print(f"Loaded {len(df):,} records with datetime index")

Loaded 120,411 records with datetime index


## Derive Additional Features Formulaically

In [2]:
# 1. `output/q4_features.csv`

# Outcome Variable: Barometric Pressure

# Begin chunk
print('Creating additional features to potentially uncover hidden patterns ...\n')

# New Derived Features 
print('Deriving "temp_difference" ...')
df['temp_difference'] = df['Air Temperature'] - df['Wet Bulb Temperature']
df['temp_difference'] = df['temp_difference'].round(2)
print(f'Last temp_difference:\n{df['temp_difference'].tail(1)}\n')

print('Deriving "temp_humidity" ...')
df['temp_humidity'] = df['Air Temperature'] * df['Humidity']
df['temp_humidity'] = df['temp_humidity'].round(2)
print(f'Last temp_humidity:\n{df['temp_humidity'].tail(1)}\n')

print('Deriving "wind_speed_squared" ...')
df['wind_speed_squared'] = df['Wind Speed'] ** 2
df['wind_speed_squared'] = df['wind_speed_squared'].round(2)
print(f'Last wind_speed_squared:\n{df['wind_speed_squared'].tail(1)}\n')

print('Deriving "wind_solar" ...')
df['wind_solar'] = df['Wind Speed'] * df['Solar Radiation']
df['wind_solar'] = df['wind_solar'].round(2)
print(f'Last wind_solar:\n{df['wind_solar'].tail(1)}\n')

print('Deriving "wind_ratio" ...')
df['wind_ratio'] = df['Wind Speed'] / (df['Maximum Wind Speed'] + 1)
df['wind_ratio'] = df['wind_ratio'].round(2)
print(f'Last wind_ratio:\n{df['wind_ratio'].tail(1)}\n')

print('Deriving "temp_category" ...')
df['temp_category'] = pd.cut(
    df['Air Temperature'],
    bins = [-np.inf, 0, 15, 30, np.inf],
    labels = ['Freezing', 'Cold', 'Moderate', 'Hot'])
print(f'Last temp_category:\n{df['temp_category'].tail(1)}\n')

# Output q4_features.csv
df.reset_index().to_csv('output/q4_features.csv', index = False)

# End chunk
print('Derivation of additional features complete! Results saved at: output/q4_features.csv')

Creating additional features to potentially uncover hidden patterns ...

Deriving "temp_difference" ...
Last temp_difference:
Measurement Timestamp
2025-12-08 16:00:00    2.4
Name: temp_difference, dtype: float64

Deriving "temp_humidity" ...
Last temp_humidity:
Measurement Timestamp
2025-12-08 16:00:00   -78.4
Name: temp_humidity, dtype: float64

Deriving "wind_speed_squared" ...
Last wind_speed_squared:
Measurement Timestamp
2025-12-08 16:00:00    0.36
Name: wind_speed_squared, dtype: float64

Deriving "wind_solar" ...
Last wind_solar:
Measurement Timestamp
2025-12-08 16:00:00    9.6
Name: wind_solar, dtype: float64

Deriving "wind_ratio" ...
Last wind_ratio:
Measurement Timestamp
2025-12-08 16:00:00    0.16
Name: wind_ratio, dtype: float64

Deriving "temp_category" ...
Last temp_category:
Measurement Timestamp
2025-12-08 16:00:00    Freezing
Name: temp_category, dtype: category
Categories (4, object): ['Freezing' < 'Cold' < 'Moderate' < 'Hot']

Derivation of additional features comp

## Deriving Additional Rolling Features

In [3]:
# 2. `output/q4_rolling_features.csv`

# Begin chunk
print('Creating rolling features to potentially uncover hidden patterns ...\n')

# New Derived Features
print('Deriving "air_temp_rolling_24h" ...')
df['air_temp_rolling_24h'] = df.groupby('Station Name')['Air Temperature'].transform(lambda x: x.rolling(window = 24).mean())
df['air_temp_rolling_24h'] = df['air_temp_rolling_24h'].round(2)
print(f'Last air_temp_rolling_24h:\n{df['air_temp_rolling_24h'].tail(1)}\n')

print('Deriving "humidity_rolling_24h" ... ')
df['humidity_rolling_24h'] = df.groupby('Station Name')['Humidity'].transform(lambda x: x.rolling(window = 24).mean())
df['humidity_rolling_24h'] = df['humidity_rolling_24h'].round(2)
print(f'Last humidity_rolling_24h:\n{df['humidity_rolling_24h'].tail(1)}\n')

print('Deriving "wind_speed_rolling_7h" ...')
df['wind_speed_rolling_7h'] = df.groupby('Station Name')['Wind Speed'].transform(lambda x: x.rolling(window = 7).mean())
df['wind_speed_rolling_7h'] = df['wind_speed_rolling_7h'].round(2)
print(f'Last wind_speed_rolling_7h:\n{df['wind_speed_rolling_7h'].tail(1)}\n')


print('Deriving "solar_radiation_7h" ...')
df['solar_radiation_7h'] = df.groupby('Station Name')['Solar Radiation'].transform(lambda x: x.rolling(window = 7).mean())
df['solar_radiation_7h'] = df['solar_radiation_7h'].round(2)
print(f'Last solar_radiation_7h:\n{df['solar_radiation_7h'].tail(1)}\n')

# Output q4_rolling_features.csv
df.reset_index().to_csv('output/q4_rolling_features.csv', index = False)

# End chunk
print('Derivation of rolling features complete! Data saved at: output/q4_rolling_features.csv')

Creating rolling features to potentially uncover hidden patterns ...

Deriving "air_temp_rolling_24h" ...
Last air_temp_rolling_24h:
Measurement Timestamp
2025-12-08 16:00:00   -1.86
Name: air_temp_rolling_24h, dtype: float64

Deriving "humidity_rolling_24h" ... 
Last humidity_rolling_24h:
Measurement Timestamp
2025-12-08 16:00:00    64.54
Name: humidity_rolling_24h, dtype: float64

Deriving "wind_speed_rolling_7h" ...
Last wind_speed_rolling_7h:
Measurement Timestamp
2025-12-08 16:00:00    3.0
Name: wind_speed_rolling_7h, dtype: float64

Deriving "solar_radiation_7h" ...
Last solar_radiation_7h:
Measurement Timestamp
2025-12-08 16:00:00    85.86
Name: solar_radiation_7h, dtype: float64

Derivation of rolling features complete! Data saved at: output/q4_rolling_features.csv


## List of Derived Features

In [4]:
# 3. `output/q4_feature_list.txt`

# Begin chunk
print('Creating a text file containing names of each feature derived ...')

# Create feature list
feature_list = [
    'temp_difference', 'temp_humidity', 'wind_speed_squared', 'wind_solar', 'wind_ratio',
    'temp_category', 'air_temp_rolling_24h', 'humidity_rolling_24h', 'wind_speed_rolling_7h', 'solar_radiation_7h'
]

# Store feature list
with open('output/q4_feature_list.txt', 'w') as f:
    for feature in feature_list:
        f.write(f'{feature}\n')

# End chunk
print('Creation of feature list text file complete! Features saved at: output/q4_feature_list.txt')

Creating a text file containing names of each feature derived ...
Creation of feature list text file complete! Features saved at: output/q4_feature_list.txt


---

## Objective

Create derived features, perform time-based aggregations, and calculate rolling windows for time series analysis.

**Time Series Note:** Rolling windows are essential for time series data. They capture temporal dependencies (e.g., 7-hour rolling mean captures short-term patterns). See **Lecture 09** for time series rolling window operations. For hourly data, common window sizes are 7-24 hours (capturing daily patterns). Use pandas `rolling()` method with `window` parameter to specify the number of periods.

---

## Required Artifacts

You must create exactly these 3 files in the `output/` directory:

### 1. `output/q4_features.csv`
**Format:** CSV file
**Content:** Dataset with all derived features added
**Requirements:**
- All original columns from Q3
- All new derived features added as columns
- **No index column** (save with `index=False`)

### 2. `output/q4_rolling_features.csv`
**Format:** CSV file
**Content:** Dataset with rolling window features
**Required Columns:**
- Original datetime column
- At least one rolling window calculation column (e.g., `water_temp_rolling_7h`, `air_temp_rolling_24h`)

**Requirements:**
- Must include at least one rolling window calculation
- Rolling window names should be descriptive (e.g., `temp_rolling_7h` for 7-hour rolling mean)
- **No index column** (save with `index=False`)

**Example columns:**
```csv
Measurement Timestamp,wind_speed_rolling_7h,humidity_rolling_24h,pressure_rolling_7h
2022-01-01 00:00:00,6.8,65.2,1013.5
2022-01-01 01:00:00,6.9,65.3,1013.6
...
```

**Note:** The example shows rolling windows of predictor variables (wind speed, humidity, pressure), not the target variable. If you're predicting Air Temperature, do NOT create rolling windows of Air Temperature - this causes data leakage.

### 3. `output/q4_feature_list.txt`
**Format:** Plain text file
**Content:** List of new features created (one per line)
**Requirements:**
- One feature name per line
- No extra text, just feature names
- Include all derived features, rolling features, and categorical features created

**Example format:**
```
temp_difference
temp_ratio
wind_speed_squared
comfort_index
water_temp_rolling_7h
air_temp_rolling_24h
wind_speed_rolling_7h
temp_category
wind_category
```

---

## Requirements Checklist

- [ ] Derived features created (differences, ratios, interactions, etc.)
- [ ] Time-based aggregations performed (by hour, day, month, etc.) - optional but recommended
- [ ] At least one rolling window calculation (rolling mean, rolling median, etc.)
- [ ] Categorical features created (if applicable)
- [ ] Feature list documented
- [ ] All 3 required artifacts saved with exact filenames

---

## Your Approach

1. **Create derived features** - Differences, ratios, interactions between variables (watch for division by zero)
2. **Calculate rolling windows** - Use `.rolling()` on predictor variables to capture temporal patterns

   ⚠️ **Data Leakage Warning:** Do not create ANY features that use your target variable - this includes rolling windows, differences, ratios, or interactions involving the target. For example, if predicting Air Temperature, do not create `air_temp * humidity` or `air_temp - wet_bulb`. Only derive features from other predictor variables.

3. **Create categorical features** - Bin continuous variables if useful (optional)
4. **Check for infinity values** - Ratios can produce infinity; replace with NaN and handle appropriately
5. **Document and save** - Remember to `reset_index()` before saving CSVs

---

## Decision Points

- **Derived features:** What relationships might be useful? Temperature differences? Ratios? Interactions between variables?
- **Rolling windows:** What window size makes sense? 7 hours? 24 hours? Consider the temporal scale of your data. For hourly data, 7-24 hours captures daily patterns.
- **Time-based aggregations:** Aggregate by hour? Day? Week? What temporal granularity is useful for your analysis?

---

## Checkpoint

After Q4, you should have:
- [ ] Derived features created
- [ ] At least one rolling window calculation
- [ ] Feature list documented
- [ ] All 3 artifacts saved: `q4_features.csv`, `q4_rolling_features.csv`, `q4_feature_list.txt`

---

**Next:** Continue to `q5_pattern_analysis.md` for Pattern Analysis.
